# Ensemble Color Quantization Framework
This notebook implements a multi-space clustering and consensus-weighting mechanism to extract an optimal K-color palette from a given image. 



## The pipeline includes:
1. Base palette extraction across RGB, LAB, HSV, and YUV color spaces.
2. Perceptual model evaluation using HaarPSI.
3. Consensus weighting and meta-clustering in CIELAB space.

# Initialization & Global Settings
In this block, we import the required libraries and set up our primary evaluation metrics. 
> **User Configurable:** You can change `K` to adjust the final number of colors in the palette.

In [ ]:
# Standard Library
import math
import time
import warnings
warnings.filterwarnings("ignore")

# Third-Party Libraries (Core Data & Processing)
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances

# Image Processing & Metrics
from brisque import BRISQUE
from sewar.full_ref import msssim
from skimage import color, data, img_as_float, img_as_ubyte
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# Deep Learning Frameworks
import lpips
from piqa import HaarPSI
import torch

## Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
brisque_metric = BRISQUE(url=False)
haarpsi_metric = HaarPSI().to(device)
lpips_metric = lpips.LPIPS(net='alex', verbose=False).to(device)

In [ ]:
K = 16          # Target number of colors in the final palette
N_MODELS = 4    # Number of color spaces used (RGB, LAB, HSV, YUV)
np.random.seed(42)

# Image Loading
Provide the path to the target image. If the image is not found, the script will default to a sample image from the `skimage` library.
> **User Configurable:** Change `IMAGE_PATH` to point to your local file (e.g., `'../data/kodim01.png'`).

In [ ]:
IMAGE_PATH = '../data/kodim01.png' # <---- put your image path here
image_np = cv2.imread(IMAGE_PATH)

In [ ]:
image_np = cv2.cvtColor(image_np, cv2.COLOR_BGR2RGB)

image_float = img_as_float(image_np)
h, w, c = image_float.shape
print(f"Image loaded successfully. Shape: {h}x{w}x{c}")

# Helper & Evaluation Functions
These functions compute standard and perceptual image quality metrics to evaluate the generated palettes.

In [ ]:
def to_uint8(img):
    if img.dtype == np.uint8:
        return img
    return img_as_ubyte(np.clip(img, 0, 1))

In [ ]:
def calculate_mmpe(original_image_rgb, palette_rgb, use_lab=True, batch_size=100000):
    orig_img_norm = original_image_rgb.astype(np.float32) / 255.0
    palette_norm = palette_rgb.astype(np.float32) / 255.0

    if use_lab:
        img_flat = color.rgb2lab(orig_img_norm).reshape(-1, 3)
        pal_target = color.rgb2lab(np.array([palette_norm]))[0]
    else:
        img_flat = orig_img_norm.reshape(-1, 3)
        pal_target = palette_norm

    num_pixels = img_flat.shape[0]
    total_min_distance = 0.0

    for i in range(0, num_pixels, batch_size):
        batch_pixels = img_flat[i:i + batch_size]
        distances = cdist(batch_pixels, pal_target, metric='euclidean')
        min_distances = np.min(distances, axis=1)
        total_min_distance += np.sum(min_distances)

    mean_min_distance = total_min_distance / num_pixels
    return mean_min_distance

In [ ]:
def evaluate_quality(img_orig, img_recon):
    if img_orig.dtype != np.uint8:
        img_orig = img_as_ubyte(img_orig)
    if img_recon.dtype != np.uint8:
        img_recon = np.clip(img_recon, 0, 1)
        img_recon = img_as_ubyte(img_recon)

    psnr_val = peak_signal_noise_ratio(img_orig, img_recon)

    min_side = min(img_orig.shape[0], img_orig.shape[1])
    win_size = min(7, min_side) if min_side >= 7 else min_side
    ssim_val = structural_similarity(
        img_orig, img_recon, channel_axis=-1, data_range=255, win_size=win_size
    )

    try:
        msssim_out = msssim(img_orig, img_recon) if msssim is not None else ssim_val
        msssim_val = msssim_out[0] if isinstance(msssim_out, tuple) else msssim_out
        if isinstance(msssim_val, complex):
            msssim_val = msssim_val.real
    except:
        msssim_val = ssim_val

    try:
        brisque_val = brisque_metric.score(img_recon) if brisque_metric else float('nan')
    except:
        brisque_val = float('nan')

    lab_orig = color.rgb2lab(img_orig / 255.0)
    lab_recon = color.rgb2lab(img_recon / 255.0)
    delta_e_map = color.deltaE_ciede2000(lab_orig, lab_recon)
    mean_delta_e = np.mean(delta_e_map)

    extracted_palette = np.unique(img_recon.reshape(-1, 3), axis=0)
    mmpe_val = calculate_mmpe(img_orig, extracted_palette, use_lab=True)
    num_unique_colors = len(extracted_palette)

    orig_t_01 = None
    recon_t_01 = None
    if torch is not None and (haarpsi_metric is not None or lpips_metric is not None):
        orig_t_01 = torch.from_numpy(img_orig.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
        recon_t_01 = torch.from_numpy(img_recon.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)

    if haarpsi_metric is not None and orig_t_01 is not None:
        try:
            with torch.no_grad():
                haarpsi_val = haarpsi_metric(orig_t_01, recon_t_01).item()
        except Exception:
            haarpsi_val = float('nan')
    else:
        haarpsi_val = float('nan')

    if lpips_metric is not None and orig_t_01 is not None:
        try:
            orig_t_lpips = orig_t_01 * 2.0 - 1.0
            recon_t_lpips = recon_t_01 * 2.0 - 1.0
            with torch.no_grad():
                lpips_val = lpips_metric(orig_t_lpips, recon_t_lpips).item()
        except Exception:
            lpips_val = float('nan')
    else:
        lpips_val = float('nan')

    return {
        'PSNR': psnr_val,
        'SSIM': ssim_val,
        'MS-SSIM': msssim_val,
        'HaarPSI': haarpsi_val,
        'LPIPS': lpips_val,
        'BRISQUE': brisque_val,
        'DeltaE_2000': mean_delta_e,
        'MMPE': mmpe_val,
        'Unique Colors': num_unique_colors
    }

In [ ]:
def luminance_of_color(rgb):
    r, g, b = rgb
    return 0.2126 * r + 0.7152 * g + 0.0722 * b

In [ ]:
def apply_kmeans(pixels, n_clusters):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
    labels = kmeans.fit_predict(pixels)
    return kmeans.cluster_centers_, labels

# Base Model Generation (Multi-Space)
The ensemble pipeline begins by independently clustering the image in four distinct spaces: RGB, CIELAB, HSV, and YUV. This ensures diverse perceptual characteristics are captured.

In [ ]:
print("\n--- Running Ensemble System ---")
ensemble_start_time = time.time()

spaces = {
    'RGB': {'to': lambda x: x, 'from': lambda x: x},
    'LAB': {'to': color.rgb2lab, 'from': color.lab2rgb},
    'HSV': {'to': color.rgb2hsv, 'from': color.hsv2rgb},
    'YUV': {'to': color.rgb2yuv, 'from': color.yuv2rgb}
}

candidates_rgb_list = []
model_reconstructions = {}
model_palettes_rgb = {}

for name, funcs in spaces.items():
    img_space = funcs['to'](image_float)
    pixels_space = img_space.reshape(-1, 3)

    centers_space, labels_space = apply_kmeans(pixels_space, K)
    recon_space = centers_space[labels_space].reshape(h, w, 3)
    recon_rgb = np.clip(funcs['from'](recon_space), 0, 1)
    model_reconstructions[name] = recon_rgb

    centers_space_img = centers_space.reshape(1, K, 3)
    centers_rgb_converted = np.clip(funcs['from'](centers_space_img), 0, 1).reshape(K, 3)

    model_palettes_rgb[name] = centers_rgb_converted
    candidates_rgb_list.append(centers_rgb_converted)

candidates_rgb = np.vstack(candidates_rgb_list)  # Resulting in 4*K candidates (4 spaces * K colors)

# Model Evaluation (HaarPSI)
We calculate a unified initial weight for each base model. The primary metric is HaarPSI.

In [ ]:
base_weights = {}
print("  Evaluating base models with HaarPSI:")

for name, recon_img in model_reconstructions.items():
    metrics = evaluate_quality(image_np, recon_img)
    
    if not np.isnan(metrics['HaarPSI']):
        weight_val = metrics['HaarPSI']
    else:
        print(f"    Warning: HaarPSI is NaN for {name}. Using uniform fallback weight.")
        weight_val = 1.0  
        
    base_weights[name] = weight_val
    print(f"    {name}: HaarPSI={metrics['HaarPSI']:.4f}, Weight={weight_val:.6f}")

model_order = ['RGB', 'LAB', 'HSV', 'YUV']
initial_weights = np.repeat([base_weights[m] for m in model_order], K)
candidate_sources = np.repeat(model_order, K)

# Consensus Weighting
We identify geometrically proximate candidate colors within the RGB space. If multiple models propose similar colors (within the threshold $\tau$), those colors receive a boosted consensus weight.
> **User Configurable:** `threshold = 0.05` manages the geometric proximity boundary for color consensus.

In [ ]:
dist_matrix = pairwise_distances(candidates_rgb, candidates_rgb)
threshold = 0.05

consensus_weights_raw = np.zeros(N_MODELS * K)
neighbor_counts = np.zeros(N_MODELS * K, dtype=int)
neighbor_indices_list = []

for i in range(N_MODELS * K):
    neighbors = np.where(dist_matrix[i] < threshold)[0]
    neighbor_indices_list.append(neighbors)
    neighbor_counts[i] = len(neighbors)
    consensus_weights_raw[i] = np.sum(initial_weights[neighbors])

if np.sum(consensus_weights_raw) > 0:
    consensus_weights = consensus_weights_raw / np.sum(consensus_weights_raw)
else:
    consensus_weights = np.ones_like(consensus_weights_raw) / len(consensus_weights_raw)

candidate_info = []
for i in range(N_MODELS * K):
    candidate_info.append({
        'index': i,
        'source': candidate_sources[i],
        'rgb': candidates_rgb[i],
        'base_model_weight': base_weights[candidate_sources[i]],
        'initial_weight': initial_weights[i],
        'neighbor_count': neighbor_counts[i],
        'consensus_weight_raw': consensus_weights_raw[i],
        'consensus_weight': consensus_weights[i],
        'neighbors': neighbor_indices_list[i]
    })

# Final Meta-Clustering & Reconstruction
The 4 * K candidate colors are transformed into the perceptually uniform CIELAB space. Weighted K-Means is executed to derive the definitive K-color palette, followed by mapping the image to these exact colors.

In [ ]:
candidates_lab = color.rgb2lab(candidates_rgb.reshape(1, N_MODELS * K, 3)).reshape(N_MODELS * K, 3)

meta_kmeans = KMeans(n_clusters=K, random_state=42, n_init='auto')
meta_kmeans.fit(candidates_lab, sample_weight=consensus_weights)

final_palette_lab = meta_kmeans.cluster_centers_
final_palette_rgb = np.clip(color.lab2rgb(final_palette_lab.reshape(1, K, 3)).reshape(K, 3), 0, 1)

pixels_lab = color.rgb2lab(image_float).reshape(-1, 3)
final_distances = pairwise_distances(pixels_lab, final_palette_lab)
final_labels = np.argmin(final_distances, axis=1)

recon_rgb_flat = final_palette_rgb[final_labels]
img_ensemble = np.clip(recon_rgb_flat.reshape(h, w, 3), 0, 1)

final_color_usage = np.bincount(final_labels, minlength=K).astype(np.float64)
if final_color_usage.sum() > 0:
    final_color_usage = final_color_usage / final_color_usage.sum()

time_ensemble = time.time() - ensemble_start_time
metrics_ensemble = evaluate_quality(image_np, img_ensemble)

# Final Performance Report
Displays a comprehensive, comparative quantitative review of the Ensemble Method.

In [ ]:
def print_metrics(model_name, metrics, exec_time):
    print(f"🔹 {model_name}:")
    print(f"   ├─ PSNR:          {metrics['PSNR']:.4f} dB")
    print(f"   ├─ SSIM:          {metrics['SSIM']:.6f}")
    print(f"   ├─ MS-SSIM:       {metrics['MS-SSIM']:.6f}")
    print(f"   ├─ HaarPSI:       {metrics['HaarPSI']:.6f}")
    print(f"   ├─ LPIPS:         {metrics['LPIPS']:.6f}")
    print(f"   ├─ BRISQUE:       {metrics['BRISQUE']:.4f}")
    print(f"   ├─ DeltaE_2000:   {metrics['DeltaE_2000']:.6f}")
    print(f"   ├─ MMPE:          {metrics['MMPE']:.6f}")
    print(f"   ├─ Unique Colors: {metrics['Unique Colors']} / {K}")
    print(f"   └─ Time:          {exec_time:.4f} sec\n")

In [ ]:
print("\n" + "=" * 70)
print("FINAL EVALUATION METRICS")
print("=" * 70)

print_metrics("Proposed Ensemble", metrics_ensemble, time_ensemble)

print("Base model weights:")
for m in model_order:
    print(f"  {m}: {base_weights[m]:.6f}")

print(f"\nConsensus threshold = {threshold}")

# Visualization Tools
Functions to display palettes, reconstructed images, and consensus tables cleanly within the notebook.

In [ ]:
def make_horizontal_palette_image(palette, patch_width=50, patch_height=50):
    palette = np.asarray(palette, dtype=np.float32)
    n = len(palette)
    canvas = np.ones((patch_height, n * patch_width, 3), dtype=np.float32)
    for i, clr in enumerate(palette):
        canvas[:, i * patch_width:(i + 1) * patch_width, :] = np.clip(clr, 0, 1)
    return canvas


In [ ]:
def show_single_image(title, image_rgb, figsize=(7, 7)):
    plt.figure(figsize=figsize)
    plt.imshow(to_uint8(image_rgb))
    plt.title(title, fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def show_palette_only(title, palette_rgb, fig_width=14, patch_width=50, patch_height=50):
    palette_img = make_horizontal_palette_image(palette_rgb, patch_width=patch_width, patch_height=patch_height)
    plt.figure(figsize=(fig_width, 2))
    plt.imshow(palette_img)
    plt.title(title, fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def show_image_and_palette_separate(title_prefix, image_rgb, palette_rgb):
    plt.figure(figsize=(7, 7))
    plt.imshow(to_uint8(image_rgb))
    plt.title(f"{title_prefix} Reconstruction", fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    palette_img = make_horizontal_palette_image(palette_rgb, patch_width=50, patch_height=50)
    plt.figure(figsize=(12, 1.8))
    plt.imshow(palette_img)
    plt.title(f"{title_prefix} Palette ({len(palette_rgb)} colors)", fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def show_candidate_palette_with_table(title, palette_rgb, info_list, patch_width=28, patch_height=70, ncols_text=2, figsize=(22, 10), sort_by=None):
    palette_rgb = np.asarray(palette_rgb, dtype=np.float32)
    n = len(palette_rgb)

    if sort_by is not None:
        order = np.argsort([x[sort_by] for x in info_list])[::-1]
        palette_rgb = palette_rgb[order]
        info_list = [info_list[i] for i in order]

    palette_img = make_horizontal_palette_image(palette_rgb, patch_width=patch_width, patch_height=patch_height)
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(2, 1, height_ratios=[1.2, 3.8])

    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(palette_img)
    ax1.set_title(title, fontsize=13, fontweight='bold')
    ax1.axis('off')

    for i in range(n):
        x_center = i * patch_width + patch_width / 2
        clr = palette_rgb[i]
        txt_color = 'white' if luminance_of_color(clr) < 0.45 else 'black'
        ax1.text(x_center, patch_height / 2, f"{info_list[i]['index']:02d}", ha='center', va='center', fontsize=8, fontweight='bold', color=txt_color)

    ax2 = fig.add_subplot(gs[1])
    ax2.axis('off')

    header = "idx | src | initial_w | neigh | raw_consensus | final_consensus"
    lines = [header, "-" * len(header)]

    for info in info_list:
        line = (f"{info['index']:02d} | {info['source']:<3} | {info['initial_weight']:>9.4f} | {info['neighbor_count']:>5d} | {info['consensus_weight_raw']:>13.4f} | {info['consensus_weight']:>15.6f}")
        lines.append(line)

    n_lines = len(lines)
    rows_per_col = int(math.ceil(n_lines / ncols_text))
    x_positions = np.linspace(0.01, 0.99, ncols_text, endpoint=True)
    y_start = 0.98
    y_step = 0.95 / rows_per_col

    for col in range(ncols_text):
        start = col * rows_per_col
        end = min((col + 1) * rows_per_col, n_lines)
        block = lines[start:end]
        for row_idx, txt in enumerate(block):
            ax2.text(x_positions[col], y_start - row_idx * y_step, txt, transform=ax2.transAxes, ha='left', va='top', fontsize=9, family='monospace')

    plt.tight_layout()
    plt.show()

In [ ]:
def show_final_palette_with_usage_table(title, palette_rgb, usage_weights, patch_width=60, patch_height=70, figsize=(16, 5)):
    palette_rgb = np.asarray(palette_rgb, dtype=np.float32)
    usage_weights = np.asarray(usage_weights, dtype=np.float64)
    n = len(palette_rgb)

    palette_img = make_horizontal_palette_image(palette_rgb, patch_width=patch_width, patch_height=patch_height)
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(2, 1, height_ratios=[1.4, 1.6])

    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(palette_img)
    ax1.set_title(title, fontsize=13, fontweight='bold')
    ax1.axis('off')

    for i in range(n):
        x_center = i * patch_width + patch_width / 2
        clr = palette_rgb[i]
        txt_color = 'white' if luminance_of_color(clr) < 0.45 else 'black'
        ax1.text(x_center, patch_height / 2, f"{i:02d}", ha='center', va='center', fontsize=9, fontweight='bold', color=txt_color)

    ax2 = fig.add_subplot(gs[1])
    ax2.axis('off')

    header = "idx | usage_weight"
    lines = [header, "-" * len(header)]
    for i, w in enumerate(usage_weights):
        lines.append(f"{i:02d} | {w:.6f}")

    rows = int(math.ceil(len(lines) / 4))
    x_positions = [0.05, 0.30, 0.55, 0.80]
    y_start = 0.95
    y_step = 0.90 / rows

    for col in range(4):
        start = col * rows
        end = min((col + 1) * rows, len(lines))
        block = lines[start:end]
        for row_idx, txt in enumerate(block):
            ax2.text(x_positions[col], y_start - row_idx * y_step, txt, transform=ax2.transAxes, ha='left', va='top', fontsize=10, family='monospace')

    plt.tight_layout()
    plt.show()

In [ ]:
def show_original_vs_ensemble(original_img, ensemble_img, title1="Original Image", title2="Ensemble Output", figsize=(14, 7)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    axes[0].imshow(to_uint8(original_img))
    axes[0].set_title(title1, fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # نمایش عکس خروجی Ensemble
    axes[1].imshow(to_uint8(ensemble_img))
    axes[1].set_title(title2, fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visual Analysis
Invokes the visualization toolkit to visually review structural integrity and color preservation across the process.

In [ ]:
show_original_vs_ensemble(image_float, img_ensemble)

In [ ]:
# optional 

# show_single_image("Original Image", image_float, figsize=(7, 7))

# # Display each base space output
# for name in ['RGB', 'LAB', 'HSV', 'YUV']:
#     show_image_and_palette_separate(
#         title_prefix=name,
#         image_rgb=model_reconstructions[name],
#         palette_rgb=model_palettes_rgb[name]
#     )

# # 64-Color Candidate Palette & Raw Tables
# show_candidate_palette_with_table(
#     title=f"Candidate Palette (64 colors) + Weight Table | threshold={threshold}",
#     palette_rgb=candidates_rgb,
#     info_list=candidate_info,
#     patch_width=28, patch_height=70, ncols_text=2, figsize=(24, 10), sort_by=None
# )

# # Candidates Sorted By Consensus Importance
# show_candidate_palette_with_table(
#     title=f"Candidate Palette Sorted by Final Consensus Weight | threshold={threshold}",
#     palette_rgb=candidates_rgb,
#     info_list=candidate_info,
#     patch_width=28, patch_height=70, ncols_text=2, figsize=(24, 10), sort_by='consensus_weight'
# )

# # Final Ensemble Reconstruction & Palette Usage Weights
# show_single_image("Final Ensemble Output", img_ensemble, figsize=(7, 7))
# show_final_palette_with_usage_table(
#     title="Final Ensemble Palette (16 colors) + Usage Weights",
#     palette_rgb=final_palette_rgb,
#     usage_weights=final_color_usage,
#     patch_width=55, patch_height=70, figsize=(16, 5)
# )